# Scenario 4: Developer Productivity with Claude

**Domains tested:** Tool Design & MCP (D2 -- 18%), Claude Code Config (D3 -- 20%), Agentic Architecture (D1 -- 27%)

---

## Scenario Context

You are a senior engineer at **NexaBank**, a fintech company with a 2M-line monorepo spanning Go microservices, a React frontend, Python ML pipelines, and Terraform infrastructure. A new hire needs to onboard quickly using Claude Code as their codebase exploration assistant.

**Production requirements:**
- The agent must help engineers navigate the codebase using built-in tools (Read, Grep, Glob, Bash)
- Tool selection must be efficient: searching file names vs searching file contents require different tools
- MCP servers provide additional capabilities (database schema viewer, deployment status)
- Long exploration sessions must manage context to avoid degradation
- The agent must handle incremental exploration: start broad, then narrow down

**Built-in tools available in Claude Code:**
- `Read` -- read file contents
- `Write` -- create or overwrite files
- `Edit` -- make targeted edits to existing files
- `Bash` -- execute shell commands
- `Grep` -- search file contents by regex pattern (ripgrep-based)
- `Glob` -- find files by name/path pattern

---

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

---

### Question 1 (D2 + D3)

A new developer asks Claude Code: "Where is the authentication middleware defined?" Claude Code uses `Grep` to search for `authentication middleware` in all files. The search returns no results because the actual code uses `authMiddleware` as the identifier.

What should Claude Code have done instead?

- **A)** Used `Glob` to search for files named `*auth*middleware*`
- **B)** Used `Bash` to run `find . -name '*auth*'`
- **C)** Used `Grep` with a broader regex pattern like `auth.*[Mm]iddleware` to account for naming variations
- **D)** Asked the developer for the exact function name before searching

In [ ]:
q1 = {
    "correct": "C",
    "explanation": (
        "Grep with a flexible regex is the right tool when you're searching for content (function "
        "definitions, variable names) but don't know the exact casing or naming convention. "
        "The pattern auth.*[Mm]iddleware matches authMiddleware, AuthMiddleware, auth_middleware, etc. "
        "Option A (Glob) searches file NAMES, not file CONTENTS -- it would find files named "
        "auth_middleware.go but not a function defined inside router.go. Option B uses Bash for "
        "something Glob does better. Option D breaks the flow of exploration. "
        "This tests Domain 2: choosing the right tool for content search vs path search."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D2)

A developer asks: "Find all the GraphQL schema files in the project." Which tool is the most appropriate?

- **A)** `Grep` with pattern `type Query` to find GraphQL definitions
- **B)** `Glob` with pattern `**/*.graphql` to find files by extension
- **C)** `Bash` with `ls -R | grep graphql`
- **D)** `Read` on the project root to see the directory listing

In [ ]:
q2 = {
    "correct": "B",
    "explanation": (
        "Glob is designed for finding files by name or path pattern. When you need to find files "
        "by extension (*.graphql), Glob is the correct tool. Option A (Grep) searches file CONTENTS "
        "-- it would find GraphQL definitions inside .ts or .go files but miss schema-only .graphql "
        "files that don't contain 'type Query'. It also returns content matches, not file paths. "
        "Option C works but Glob is purpose-built and more efficient. Option D reads a single "
        "directory, not recursively. Key distinction: Glob = find by path/name, Grep = find by "
        "content. This is a critical Domain 2 concept the exam tests repeatedly."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D2 + D3)

Your team wants to add a custom MCP server that lets Claude Code query the production database schema (read-only). The MCP server configuration goes in `.claude/settings.json`. A junior developer adds it to `.claude/settings.local.json` instead.

What is the difference?

- **A)** `settings.json` is for development and `settings.local.json` is for production
- **B)** They are the same -- Claude Code merges both files
- **C)** `settings.local.json` is a deprecated format that Claude Code ignores
- **D)** `settings.json` is committed to the repo and shared with the team; `settings.local.json` is gitignored and applies only to the individual developer's machine

In [ ]:
q3 = {
    "correct": "D",
    "explanation": (
        "settings.json is checked into the repo (shared team config), while settings.local.json "
        "is gitignored (personal config). Both are loaded and merged, but they serve different "
        "purposes. Team-wide MCP servers (like the database schema viewer) should go in "
        "settings.json so every developer gets the same tools. Personal MCP servers or API keys "
        "should go in settings.local.json. If the junior developer puts the team server in "
        "settings.local.json, other team members won't have it. This tests Domain 3: "
        "understanding the Claude Code configuration hierarchy."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D1 + D2)

An engineer asks Claude Code to explore the payment processing flow. Claude Code reads 15 files in sequence, and by the time it summarizes, it has forgotten details from the first files it read. What is the best exploration strategy?

- **A)** Read all files first, then summarize at the end
- **B)** Use an incremental exploration strategy: start with Grep/Glob to identify the relevant files, read the entry point, follow call chains depth-first while summarizing findings at each step before moving to the next file
- **C)** Use a larger model with more context
- **D)** Ask the engineer to specify exactly which files to read

In [ ]:
q4 = {
    "correct": "B",
    "explanation": (
        "Incremental exploration with intermediate summaries is the correct pattern for codebase "
        "navigation. The strategy: (1) Glob/Grep to identify candidate files, (2) Read the entry "
        "point, (3) note key function calls and data flow, (4) follow the most important call "
        "chain, (5) summarize findings before reading the next file. This keeps the working "
        "context focused. Option A dumps everything into context at once, causing attention "
        "degradation. Option C doesn't fix the exploration strategy. Option D defeats the purpose "
        "of an exploration assistant. This tests Domain 1 (agentic exploration strategy) and "
        "Domain 2 (tool selection for efficient exploration)."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D3 + D5)

After a 2-hour exploration session with Claude Code, the engineer notices that Claude Code is repeating tool calls it already made earlier in the conversation. The same files are being re-read, and the same Grep searches are being re-run. What is happening?

- **A)** Claude Code has a bug that doesn't cache tool results
- **B)** The model's temperature is too high, causing random behavior
- **C)** The engineer is asking the same questions repeatedly
- **D)** The conversation context has grown so large that the model loses track of what it already explored -- earlier tool results are diluted by more recent conversation turns

In [ ]:
q5 = {
    "correct": "D",
    "explanation": (
        "Long conversations cause context dilution. As tool results accumulate, earlier results "
        "get proportionally less attention. The model may 'forget' that it already read a file "
        "or ran a search. Mitigations: (1) use /compact to summarize old conversation context, "
        "(2) start new sessions for new exploration topics, (3) explicitly reference earlier "
        "findings when building on them. Option A is unlikely -- Claude Code does not cache "
        "tool results between turns, but the issue is attention, not caching. Option B is not "
        "the cause. Option C doesn't match the described behavior. This tests Domain 5: "
        "context management in long-running sessions."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D2 + D1)

You want Claude Code to help engineers understand API dependencies between microservices. The codebase has 30 Go microservices, each with HTTP clients calling other services. What is the most efficient tool-based exploration approach?

- **A)** Read every Go file in every microservice to find HTTP client calls
- **B)** Use Glob to find all `*.go` files and read each one
- **C)** Use Grep to search for HTTP client patterns (e.g., `http\.NewRequest|httpClient\.Get|httpClient\.Post`) across the entire codebase, then group results by service to build a dependency map
- **D)** Use Bash to run `go build` on each service and parse the output

In [ ]:
q6 = {
    "correct": "C",
    "explanation": (
        "Grep with a targeted regex pattern is the most efficient way to find specific code "
        "patterns across a large codebase. Searching for HTTP client invocation patterns "
        "immediately identifies which services call other services, without reading every file. "
        "Option A reads the entire codebase -- extremely wasteful. Option B finds all Go files "
        "but doesn't search their contents. Option D doesn't reveal HTTP dependencies from build "
        "output. This tests Domain 2 (efficient tool selection) and Domain 1 (systematic "
        "exploration strategy for large codebases)."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

### Question 7 (D3)

Your MCP server for database schema viewing requires a connection string that varies per developer (local vs staging). Where should this connection string be configured?

- **A)** In an environment variable that the MCP server reads at startup, referenced in `.claude/settings.json`
- **B)** In the MCP server's source code
- **C)** In `.claude/settings.json` so all developers use the same connection
- **D)** In `.claude/settings.local.json` since the connection string is developer-specific and should not be committed to the repo

In [ ]:
q7 = {
    "correct": "A",
    "explanation": (
        "The best approach is to configure the MCP server in the shared settings.json (so all "
        "developers get the tool) but have it read the connection string from an environment "
        "variable (so each developer's local setup is respected). This separates the tool "
        "definition (shared) from the credentials (local). Option B hardcodes credentials. "
        "Option C commits a connection string to the repo. Option D works but then each "
        "developer must manually configure the entire MCP server entry, not just the credential. "
        "This tests Domain 3: configuration best practices for MCP servers."
    )
}
print(f"Correct: {q7['correct']}")
print(f"\nExplanation: {q7['explanation']}")

---

## Part 2: Hands-On Build

Build a codebase exploration agent that demonstrates efficient tool selection and incremental exploration.

---

### Step 1: Simulate a codebase structure

In [ ]:
# Simulated codebase file index (as if Glob returned these)
CODEBASE_FILES = {
    # Payment service
    "services/payment/main.go": "package main\n\nfunc main() { startServer(8080) }",
    "services/payment/handler.go": (
        "package main\n\n"
        "import \"net/http\"\n\n"
        "func handlePayment(w http.ResponseWriter, r *http.Request) {\n"
        "    // Validate payment request\n"
        "    req := parsePaymentRequest(r)\n"
        "    // Check with fraud service\n"
        "    fraudResult := httpClient.Post(\"http://fraud-service:8082/check\", req)\n"
        "    if fraudResult.Risk > 0.7 {\n"
        "        http.Error(w, \"Payment flagged\", 403)\n"
        "        return\n"
        "    }\n"
        "    // Process via Stripe\n"
        "    result := stripeClient.Charge(req.Amount, req.Token)\n"
        "    // Notify order service\n"
        "    httpClient.Post(\"http://order-service:8081/confirm\", result)\n"
        "}"
    ),
    "services/payment/handler_test.go": (
        "package main\n\nfunc TestHandlePayment_ValidRequest_ReturnsSuccess(t *testing.T) {}\n"
        "func TestHandlePayment_FraudDetected_Returns403(t *testing.T) {}"
    ),
    # Order service
    "services/order/main.go": "package main\n\nfunc main() { startServer(8081) }",
    "services/order/handler.go": (
        "package main\n\n"
        "func handleOrder(w http.ResponseWriter, r *http.Request) {\n"
        "    order := parseOrder(r)\n"
        "    // Validate inventory via inventory service\n"
        "    available := httpClient.Get(\"http://inventory-service:8083/check/\" + order.ItemID)\n"
        "    if !available {\n"
        "        http.Error(w, \"Out of stock\", 409)\n"
        "        return\n"
        "    }\n"
        "    db.Save(order)\n"
        "}"
    ),
    # Fraud service
    "services/fraud/main.go": "package main\n\nfunc main() { startServer(8082) }",
    "services/fraud/checker.go": (
        "package main\n\n"
        "func checkFraud(w http.ResponseWriter, r *http.Request) {\n"
        "    req := parseFraudRequest(r)\n"
        "    // ML model scoring\n"
        "    score := mlClient.Predict(req)\n"
        "    json.NewEncoder(w).Encode(FraudResult{Risk: score})\n"
        "}"
    ),
    # Config files
    "services/payment/Dockerfile": "FROM golang:1.22\nCOPY . .\nRUN go build -o payment",
    "docker-compose.yml": "version: '3'\nservices:\n  payment:\n    build: services/payment",
}


# Simulated tool functions
def sim_glob(pattern: str) -> list:
    """Simulate Glob: find files by name pattern."""
    import fnmatch
    return [f for f in CODEBASE_FILES.keys() if fnmatch.fnmatch(f, pattern)]


def sim_grep(pattern: str, file_type: str = None) -> list:
    """Simulate Grep: search file contents."""
    import re
    results = []
    for path, content in CODEBASE_FILES.items():
        if file_type and not path.endswith(file_type):
            continue
        for i, line in enumerate(content.split("\n"), 1):
            if re.search(pattern, line):
                results.append({"file": path, "line": i, "content": line.strip()})
    return results


def sim_read(path: str) -> str:
    """Simulate Read: read file contents."""
    return CODEBASE_FILES.get(path, f"ERROR: File not found: {path}")


print(f"Simulated codebase: {len(CODEBASE_FILES)} files")
print("Tools available: sim_glob, sim_grep, sim_read")

### Step 2: Demonstrate Grep vs Glob selection

In [ ]:
print("=== Tool Selection Demo ===")
print()

# Task 1: Find all Go test files (path-based -> Glob)
print("Task: Find all Go test files")
print("Correct tool: Glob (searching by file name pattern)")
test_files = sim_glob("**/*_test.go")
print(f"  Result: {test_files}")

print()

# Task 2: Find where httpClient.Post is called (content-based -> Grep)
print("Task: Find all inter-service HTTP calls")
print("Correct tool: Grep (searching file contents)")
http_calls = sim_grep(r"httpClient\.(Post|Get)")
print(f"  Results:")
for r in http_calls:
    print(f"    {r['file']}:{r['line']} -- {r['content']}")

print()

# Task 3: Find all Dockerfiles (path-based -> Glob)
print("Task: Find all Dockerfiles")
print("Correct tool: Glob (searching by file name)")
dockerfiles = sim_glob("**/Dockerfile")
print(f"  Result: {dockerfiles}")

print()

# Task 4: Find where Stripe is used (content-based -> Grep)
print("Task: Find where Stripe integration is used")
print("Correct tool: Grep (searching for code references)")
stripe_refs = sim_grep(r"stripe", ".go")
print(f"  Results:")
for r in stripe_refs:
    print(f"    {r['file']}:{r['line']} -- {r['content']}")

### Step 3: Build incremental exploration with Claude

In [ ]:
EXPLORATION_TOOLS = [
    {
        "name": "glob",
        "description": "Find files by name/path pattern. Use for: finding files by extension, locating config files, discovering directory structure. Do NOT use for searching file contents.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Glob pattern (e.g., **/*.go, **/Dockerfile)"}
            },
            "required": ["pattern"]
        }
    },
    {
        "name": "grep",
        "description": "Search file contents by regex pattern. Use for: finding function definitions, tracking variable usage, locating imports, finding inter-service calls. Do NOT use for finding files by name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Regex pattern to search for"},
                "file_type": {"type": "string", "description": "Optional file extension filter (e.g., .go, .ts)"}
            },
            "required": ["pattern"]
        }
    },
    {
        "name": "read_file",
        "description": "Read the full contents of a file. Use after Glob/Grep to read specific files of interest.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "File path to read"}
            },
            "required": ["path"]
        }
    }
]

TOOL_DISPATCH = {
    "glob": lambda args: sim_glob(args["pattern"]),
    "grep": lambda args: sim_grep(args["pattern"], args.get("file_type")),
    "read_file": lambda args: sim_read(args["path"]),
}


def explore_codebase(question: str, max_turns: int = 8):
    """Agentic codebase exploration loop."""
    system = """You are a codebase exploration assistant. Help the developer understand the codebase.

EXPLORATION STRATEGY:
1. Start broad: use Glob to understand directory structure, Grep to find entry points
2. Narrow down: Read specific files of interest
3. Follow call chains: when you find a function call, search for its definition
4. Summarize incrementally: note your findings after each step before continuing

TOOL SELECTION:
- Glob: find files by NAME/PATH (extensions, directories)
- Grep: find code by CONTENT (function definitions, imports, patterns)
- Read: read a SPECIFIC file you've already identified

Always explain which tool you're choosing and WHY before using it."""

    messages = [{"role": "user", "content": question}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL, max_tokens=2048,
            system=system, tools=EXPLORATION_TOOLS,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  [{block.name}] {json.dumps(block.input)}")
                    result = TOOL_DISPATCH[block.name](block.input)
                    print(f"  -> {json.dumps(result)[:200]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
                elif hasattr(block, "text") and block.text.strip():
                    print(f"  [thinking] {block.text[:200]}")
            messages.append({"role": "user", "content": tool_results})
        else:
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\n[SUMMARY]\n{block.text}")
            return

    print("Max turns reached.")


print("Exploration agent ready.")

In [ ]:
print("=== Exploration: Payment Flow ===")
print("Question: How does the payment processing flow work? What services does it depend on?\n")

explore_codebase("How does the payment processing flow work? What services does it depend on?")

---

## Part 3: Failure Injections

---

### Failure 1: Wrong tool selection

In [ ]:
print("=== FAILURE INJECTION: Using Glob when Grep is needed ===")
print()

# Developer asks: "Find where the fraud check is called"
# WRONG: Using Glob (finds files by name, not content)
print("Task: Find where the fraud check is called")
print("\nWRONG approach (Glob):")
wrong_result = sim_glob("**/*fraud*")
print(f"  Glob('**/*fraud*') -> {wrong_result}")
print("  This finds fraud SERVICE files, but not where fraud is CALLED from payment service.")

print("\nRIGHT approach (Grep):")
right_result = sim_grep(r"fraud")
print(f"  Grep('fraud') ->")
for r in right_result:
    print(f"    {r['file']}:{r['line']} -- {r['content']}")
print("  This finds the actual fraud check CALL in payment/handler.go.")

print("\n--- DIAGNOSIS ---")
print("Glob finds files BY NAME. Grep finds code BY CONTENT.")
print("'Where is X called?' = content search = Grep.")
print("'Where are the X files?' = path search = Glob.")
print("LESSON: Tool selection is a core skill for codebase exploration (Domain 2).")

### Failure 2: Reading too many files at once

In [ ]:
print("=== FAILURE INJECTION: Bulk reading without targeting ===")
print()

# Read every file in the codebase at once
print("WRONG approach: Read all files, then try to understand")
total_tokens_estimate = sum(len(content.split()) for content in CODEBASE_FILES.values())
print(f"  Total files: {len(CODEBASE_FILES)}")
print(f"  Estimated tokens: ~{total_tokens_estimate * 2}")
print("  Problem: All content dumped into context at once. Model must attend to everything.")

print("\nRIGHT approach: Targeted exploration")
print("  1. Glob('services/*/main.go') -> find entry points")
entry_points = sim_glob("services/*/main.go")
print(f"     {entry_points}")
print("  2. Read only payment/main.go -> identify the starting handler")
print("  3. Read payment/handler.go -> understand the payment flow")
print("  4. Grep for downstream calls -> find service dependencies")
print("  Total files read: 2 (not 8)")

print("\n--- DIAGNOSIS ---")
print("Incremental exploration reads only what's needed.")
print("Bulk reading wastes context and degrades attention to details.")
print("LESSON: Start broad (Glob/Grep), narrow down (Read targeted files) (Domain 1, D5).")

### Failure 3: Context exhaustion in long sessions

In [ ]:
print("=== FAILURE INJECTION: Long session context exhaustion ===")
print()
print("Simulating a long exploration session where tool results accumulate...\n")

# Simulate what the conversation looks like after many tool calls
simulated_conversation_length = []
for i in range(20):
    # Each tool call adds ~500 tokens of results
    simulated_conversation_length.append({
        "turn": i + 1,
        "action": f"Read file {i}",
        "tokens_added": 500,
        "cumulative_tokens": (i + 1) * 500
    })

print(f"After 20 tool calls: ~{20 * 500} tokens of tool results in conversation")
print(f"After 50 tool calls: ~{50 * 500} tokens of tool results in conversation")
print()
print("Symptoms of context exhaustion:")
print("  - Agent re-reads files it already read")
print("  - Agent re-runs Grep searches it already performed")
print("  - Agent gives contradictory summaries")
print("  - Agent loses track of the original question")
print()
print("Mitigations:")
print("  1. /compact -- summarize old conversation context")
print("  2. /clear -- start fresh for new topics")
print("  3. Summarize findings between exploration phases")
print("  4. Ask targeted questions instead of open-ended exploration")
print()
print("LESSON: Context quality degrades with quantity. Manage it actively (Domain 5).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| Grep vs Glob | D2 | Grep = search file contents; Glob = search file paths/names |
| Incremental exploration | D1, D2 | Start broad (Glob/Grep), narrow down (Read), summarize between steps |
| MCP server configuration | D3 | Team tools in settings.json, personal config in settings.local.json |
| settings.json vs settings.local.json | D3 | Shared vs personal; credentials via environment variables |
| Context exhaustion | D5 | Long sessions degrade quality; use /compact or /clear to manage |
| Tool descriptions | D2 | Clear descriptions of when to use each tool improve selection accuracy |